# 00_extract_api_to_raw

Extracts data from Spotify API and writes raw JSON to Workspace Files.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/auth/token_manager
%run ../../tools/api/spotify_client
%run ../../tools/io/raw_writer
%run ../../tools/monitoring/api_call_logger

In [ ]:
import uuid
from datetime import date

dbutils.widgets.text("ingestion_date", str(date.today()))
ingestion_date = dbutils.widgets.get("ingestion_date")
run_id = str(uuid.uuid4())

print(f"run_id:         {run_id}")
print(f"ingestion_date: {ingestion_date}")

In [ ]:
import json as _json

BATCH_TRACKS = 50
BATCH_AF     = 100
track_ids    = set()

# ── 1. Play history ────────────────────────────────────────────────────────────
params = {"limit": 50}
page   = 0
while True:
    data, status, req_hash = spotify_api_call("GET", "/me/player/recently-played", params=params)
    save_raw_api_call(
        run_id=run_id, method="GET", endpoint="/me/player/recently-played",
        request_url=f"{SPOTIFY_API_BASE}/me/player/recently-played",
        http_status=status, payload_json=_json.dumps(data), req_hash=req_hash,
        request_params_json=_json.dumps(params),
    )
    write_raw_json("play_history", ingestion_date, run_id, page, data)
    for item in data.get("items", []):
        track_ids.add(item["track"]["id"])
    cursor = data.get("cursors", {}).get("before")
    if not data.get("next") or not cursor:
        break
    params = {"limit": 50, "before": cursor}
    page += 1
flush_api_call_buffer()
print(f"play_history: {page + 1} pages | {len(track_ids)} unique tracks")

# ── 2. User playlists ──────────────────────────────────────────────────────────
playlist_ids = []
params       = {"limit": 50, "offset": 0}
page         = 0
while True:
    data, status, req_hash = spotify_api_call("GET", "/me/playlists", params=params)
    save_raw_api_call(
        run_id=run_id, method="GET", endpoint="/me/playlists",
        request_url=f"{SPOTIFY_API_BASE}/me/playlists",
        http_status=status, payload_json=_json.dumps(data), req_hash=req_hash,
        request_params_json=_json.dumps(params),
    )
    write_raw_json("playlists", ingestion_date, run_id, page, data)
    playlist_ids += [item["id"] for item in data.get("items", [])]
    if not data.get("next"):
        break
    params["offset"] += 50
    page += 1
flush_api_call_buffer()
print(f"playlists: {page + 1} pages | {len(playlist_ids)} playlists")

# ── 3. Playlist tracks ─────────────────────────────────────────────────────────
for pl_idx, playlist_id in enumerate(playlist_ids):
    pl_params = {"limit": 100, "offset": 0}
    pl_page   = 0
    while True:
        data, status, req_hash = spotify_api_call("GET", f"/playlists/{playlist_id}/tracks", params=pl_params)
        save_raw_api_call(
            run_id=run_id, method="GET", endpoint=f"/playlists/{playlist_id}/tracks",
            request_url=f"{SPOTIFY_API_BASE}/playlists/{playlist_id}/tracks",
            http_status=status, payload_json=_json.dumps(data), req_hash=req_hash,
            request_params_json=_json.dumps(pl_params),
        )
        write_raw_json("playlist_tracks", ingestion_date, run_id, pl_idx * 1000 + pl_page, {**data, "playlist_id": playlist_id})
        for item in data.get("items", []):
            if item.get("track") and item["track"].get("id"):
                track_ids.add(item["track"]["id"])
        if not data.get("next"):
            break
        pl_params["offset"] += 100
        pl_page += 1
flush_api_call_buffer()
print(f"playlist_tracks: {len(playlist_ids)} playlists | total tracks: {len(track_ids)}")

# ── 4. Tracks (batch 50) ───────────────────────────────────────────────────────
track_ids_list = list(track_ids)
for i in range(0, len(track_ids_list), BATCH_TRACKS):
    batch = track_ids_list[i:i + BATCH_TRACKS]
    data, status, req_hash = spotify_api_call("GET", "/tracks", params={"ids": ",".join(batch)})
    save_raw_api_call(
        run_id=run_id, method="GET", endpoint="/tracks",
        request_url=f"{SPOTIFY_API_BASE}/tracks",
        http_status=status, payload_json=_json.dumps(data), req_hash=req_hash,
    )
    write_raw_json("tracks", ingestion_date, run_id, i // BATCH_TRACKS, data)
flush_api_call_buffer()
print(f"tracks: {len(track_ids_list)} ids")

# ── 5. Artists (collect from track responses, batch 50) ───────────────────────
artist_ids = set()
for f in list_raw_files("tracks", ingestion_date, run_id):
    for t in _json.loads(dbutils.fs.head(f)).get("tracks", []):
        for a in t.get("artists", []):
            if a.get("id"):
                artist_ids.add(a["id"])

artist_ids_list = list(artist_ids)
for i in range(0, len(artist_ids_list), BATCH_TRACKS):
    batch = artist_ids_list[i:i + BATCH_TRACKS]
    data, status, req_hash = spotify_api_call("GET", "/artists", params={"ids": ",".join(batch)})
    save_raw_api_call(
        run_id=run_id, method="GET", endpoint="/artists",
        request_url=f"{SPOTIFY_API_BASE}/artists",
        http_status=status, payload_json=_json.dumps(data), req_hash=req_hash,
    )
    write_raw_json("artists", ingestion_date, run_id, i // BATCH_TRACKS, data)
flush_api_call_buffer()
print(f"artists: {len(artist_ids_list)} ids")

# ── 6. Audio features (batch 100) ─────────────────────────────────────────────
for i in range(0, len(track_ids_list), BATCH_AF):
    batch = track_ids_list[i:i + BATCH_AF]
    data, status, req_hash = spotify_api_call("GET", "/audio-features", params={"ids": ",".join(batch)})
    save_raw_api_call(
        run_id=run_id, method="GET", endpoint="/audio-features",
        request_url=f"{SPOTIFY_API_BASE}/audio-features",
        http_status=status, payload_json=_json.dumps(data), req_hash=req_hash,
    )
    write_raw_json("audio_features", ingestion_date, run_id, i // BATCH_AF, data)
flush_api_call_buffer()
print(f"audio_features: {len(track_ids_list)} ids")

print(f"\nExtraction complete. run_id={run_id}")
dbutils.notebook.exit(_json.dumps({"run_id": run_id, "ingestion_date": ingestion_date}))